## 0. 目录（建议按顺序看）

1. 总览：从脚本到 Gazebo 到机器人控制
2. 世界地图选择：`gz_world.yaml`
3. 顶层编排：`bringup_sim.launch.py`
4. 启动 Gazebo：`gazebo.launch.py`（GUI / headless）
5. 生成机器人+启动控制节点：`spawn_robots.launch.py`
6. ROS↔Gazebo 桥接：`ros_gz_bridge.yaml`
7. 机器人如何被“命令控制”：`rmua19_robot_base` → `ChassisController` → Ignition Transport
8. `sim_mapping.sh` 调用链：`launch_wrapper.py` 的关键行为
9. 为什么 `sim_mapping.sh` 不弹 Gazebo 窗口：根因与修复
10. 运行与验证：推荐命令

## 1. 总览：从脚本到 Gazebo 到机器人控制

核心链路：

```text
scripts/sim_mapping.sh
  └─→ scripts/launch_wrapper.py  (mode=sim_mapping)
        ├─→ [后台] ros2 launch rmu_gazebo_simulator bringup_sim.launch.py
        │     ├─→ launch/gazebo.launch.py        ← 启动 Ignition Gazebo
        │     ├─→ launch/spawn_robots.launch.py  ← 生成机器人 + 启动 rmua19_robot_base
        │     └─→ launch/referee_system.launch.py← 裁判系统桥接
        └─→ [前台/新终端] ros2 launch pb2025_nav_bringup ... (SLAM / Nav)
```

机器人控制命令的路径（以底盘为例）：

```text
ROS 话题（键盘/算法） → rmua19_robot_base 内部控制器 → Ignition Transport → Gazebo 物理引擎
```

## 2. 世界地图选择：`gz_world.yaml`

文件：`src/rmu_gazebo_simulator/rmu_gazebo_simulator/config/gz_world.yaml`

它决定两件事：
- **当前加载哪个世界**（`world:` 字段）
- **每个世界里每台机器人的出生点**（`robots: <world>: <robot>: x/y/z/yaw`）

例如当前：

```yaml
world: "rmuc_2026"
```

`bringup_sim.launch.py` 会根据它拼接世界 SDF 文件路径：

```text
rmu_gazebo_simulator/resource/worlds/{world}_world.sdf
```

In [ ]:
# 可选：快速查看当前选中世界与机器人出生点（不改任何东西）
from pathlib import Path
import yaml

ws = Path('/home/abc/rm_code/2026_1_21/ros2_ws')
p = ws / 'src/rmu_gazebo_simulator/rmu_gazebo_simulator/config/gz_world.yaml'
data = yaml.safe_load(p.read_text())
print('world =', data.get('world'))
robots = data.get('robots', {}).get(data.get('world'), {})
print('robots in world:', list(robots.keys()))
for name, pose in robots.items():
    print(name, 'pose=', {k: pose.get(k) for k in ['x_pose','y_pose','z_pose','yaw']})

## 3. 顶层编排：`bringup_sim.launch.py`

文件：`src/rmu_gazebo_simulator/rmu_gazebo_simulator/launch/bringup_sim.launch.py`

职责：
1. 读取 `gz_world.yaml`（获得 `selected_world`）
2. 计算 `world_sdf_path`
3. include：
   - `gazebo.launch.py`（启动 Ignition Gazebo）
   - `spawn_robots.launch.py`（生成机器人、启动控制节点、启动桥接）
   - `referee_system.launch.py`（裁判系统桥接与模拟）

关键参数：
- `use_gui`：默认 **true**（是否启动 Gazebo GUI 窗口）
- `enable_chassis_odometry_gt`：控制是否桥接 Gazebo 的真值里程计（影响导航调试）

## 4. 启动 Gazebo：`gazebo.launch.py`（GUI / headless）

文件：`src/rmu_gazebo_simulator/rmu_gazebo_simulator/launch/gazebo.launch.py`

它通过 `use_gui` 分成两条路径：

- `use_gui:=true`（默认）：
  - include `ros_gz_sim/launch/gz_sim.launch.py`
  - `gz_args`: `<world.sdf> --gui-config <gui.config>`
  - 预期行为：弹出 Ignition Gazebo 窗口

- `use_gui:=false`：
  - `gz_args`: `<world.sdf> -r -s --headless-rendering`
  - 预期行为：不弹窗口（仅服务端仿真）

此外还会：
- 设置 `GAZEBO_PLUGIN_PATH`、`IGN_GAZEBO_RESOURCE_PATH`
- 启动 `/clock` 的 `ros_gz_bridge parameter_bridge`（Gazebo 时钟 → ROS）

## 5. 生成机器人+启动控制节点：`spawn_robots.launch.py`

文件：`src/rmu_gazebo_simulator/rmu_gazebo_simulator/launch/spawn_robots.launch.py`

对 `gz_world.yaml` 当前世界中的每个机器人（例如 `red_standard_robot1`、`blue_standard_robot1`），依次做：

1. 用 `xmacro4sdf` 从 `simulation_robot.sdf.xmacro` 生成 SDF
2. 用 `ros_gz_sim create` 把 SDF 实体生成到 Ignition 世界
3. 启动 `rmoss_gz_base/rmua19_robot_base`（机器人控制核心节点，命名空间是机器人名）
4. 启动 `robot_state_publisher`（发布 TF）
5. 启动 `ros_gz_bridge/parameter_bridge`（按配置桥接传感器/里程计/关节等）
6. 调用 `ign service ... set_performer`（性能相关）

注意：之前为了调试崩溃，我们在此文件中已经加入了 `robot_base_prefix` 启动参数，用于给 `rmua19_robot_base` 加 gdb 前缀（可选）。

## 6. ROS↔Gazebo 桥接：`ros_gz_bridge.yaml`

文件：`src/rmu_gazebo_simulator/rmu_gazebo_simulator/config/ros_gz_bridge.yaml`

它是一个模板，里面用 `<robot_name>` 占位符。`spawn_robots.launch.py` 会把它替换成真实机器人名字，然后把 `config_file` 传给 `ros_gz_bridge parameter_bridge`。

典型桥接包括（方向通常是 Gazebo→ROS）：
- `/odometry`、`/joint_states`
- 相机 `image` / `camera_info`
- 激光 `scan`、点云 `points`
- IMU `imu`

## 7. 机器人如何被“命令控制”：从 ROS 命令到 Gazebo 运动

这一段是你最关心的：**在 Gazebo 世界里输入命令，机器人为什么会动**。

### 7.1 关键节点：`rmua19_robot_base`
文件：`src/rmoss_core/rmoss_gz_base/src/rmua19_robot_base_node.cpp`

它负责：
- 创建 Ignition 侧执行器（例如底盘、云台、射击）
- 创建 ROS 侧控制器（例如 `ChassisController`）
- 把 ROS 的控制命令转换成 Ignition Transport 消息发布给 Gazebo

### 7.2 底盘控制：`ChassisController`
文件：`src/rmoss_core/rmoss_gz_base/src/chassis_controller.cpp`

它订阅两类输入：
- `robot_base/chassis_cmd`（自定义 `ChassisCmd`：更高层的控制模式）
- `cmd_vel`（标准 `geometry_msgs/Twist`：直接速度控制）

它用一个 10ms 定时器调用 `update()`，最终会调用执行器：
- `chassis_actuator_->set(twist)`

### 7.3 最终执行器：Ignition Transport 发布
文件：`src/rmoss_core/rmoss_gz_base/src/gz_module/gz_chassis_actuator.cpp`

这里会把 ROS 的 `Twist` 转换成 `ignition::msgs::Twist`，并发布到 Ignition 的话题（非 ROS）：
- `/<robot_name>/cmd_vel`

Gazebo 的物理引擎/插件订阅这个 Ignition 话题，从而驱动机器人运动。

### 7.4 键盘控制工具：`test_chassis_cmd.py`
文件：`src/rmoss_core/rmoss_gz_base/rmoss_gz_base/scripts/test_chassis_cmd.py`

它读取键盘，然后发布 `ChassisCmd` 到 `chassis_cmd`（如果在某个 namespace 下运行，会变成 `/<ns>/robot_base/chassis_cmd`）。

## 8. `sim_mapping.sh` 调用链：它跟 launch 文件的关系

文件：`scripts/sim_mapping.sh`

它主要做三件事：
1. 设置一些渲染相关环境变量（NVIDIA PRIME offload 等）
2. 设置两个环境变量命令字符串：
   - `GAZEBO_CMD` 默认是：`ros2 launch rmu_gazebo_simulator bringup_sim.launch.py`
   - `SLAM_CMD` 默认是：`ros2 launch pb2025_nav_bringup rm_navigation_simulation_launch.py slam:=True`
3. `exec python3 scripts/launch_wrapper.py sim_mapping`

所以真正的编排逻辑在 `scripts/launch_wrapper.py`。

## 9. 为什么 `sim_mapping.sh` 不弹 Gazebo 窗口：根因与修复

### 9.1 现象
你执行：

```bash
bash scripts/sim_mapping.sh
```

脚本退出码是 0，但 Gazebo GUI 窗口并没出现（或瞬间出现又消失）。

### 9.2 根因（已确认）
- `launch_wrapper.py` 在 `sim_mapping` 中：
  - **Gazebo 用 `background=True` 启动**：`subprocess.Popen(..., preexec_fn=os.setsid)`，日志写到 `log/launch_wrapper_gazebo_sim.log`
  - 然后启动 SLAM：如果系统有 `gnome-terminal`，就用 `gnome-terminal --title=SLAM -- bash -c ...` 打开新终端窗口
- 关键点：`gnome-terminal` 启动新窗口后会 **立刻返回**（不会阻塞等待 SLAM 结束）
- 于是 `launch_wrapper.py` 很快 `return 0` 退出
- 退出触发 `atexit.register(bg.cleanup)`：会把刚才后台启动的 Gazebo 进程组 SIGTERM/SIGKILL 掉

**结果：Gazebo 刚启动就被脚本清理机制杀掉**，GUI 看起来像“打不开”。

证据：`log/launch_wrapper_gazebo_sim.log` 只有几十行就截断，还停留在桥接创建阶段。

### 9.3 修复（已在仓库中落地）
文件：`scripts/launch_wrapper.py`

修复思路：
- 如果处于 **多终端模式**（`gnome-terminal` 可用且 `no_new_terminal` 为 false），则在启动 SLAM/Nav 后：
  - wrapper 进程保持运行
  - 轮询等待 Gazebo 后台 PID 退出
  - 用户按 Ctrl+C 时统一清理后台进程

这样就不会脚本立刻退出导致 Gazebo 被杀。

In [ ]:
# 可选：查看 Gazebo 后台日志（用于排障）
from pathlib import Path
ws = Path('/home/abc/rm_code/2026_1_21/ros2_ws')
log = ws / 'log/launch_wrapper_gazebo_sim.log'
print(log)
if log.exists():
    print('lines=', sum(1 for _ in log.open('r', errors='ignore')))
    print('tail:')
    print(''.join(log.read_text(errors='ignore').splitlines(True)[-40:]))
else:
    print('log not found')

## 10. 运行与验证：推荐命令

### 10.1 启动仿真 + SLAM（推荐）
```bash
cd /home/abc/rm_code/2026_1_21/ros2_ws
bash scripts/sim_mapping.sh
```

### 10.2 强制打开 GUI（如果你曾经手动关过）
`bringup_sim.launch.py` 的 `use_gui` 默认就是 true；但你也可以显式指定：
```bash
GAZEBO_CMD="ros2 launch rmu_gazebo_simulator bringup_sim.launch.py use_gui:=true" \
bash scripts/sim_mapping.sh
```

### 10.3 只跑 Gazebo（不跑 SLAM）
```bash
source /opt/ros/humble/setup.bash
source /home/abc/rm_code/2026_1_21/ros2_ws/install/setup.bash
ros2 launch rmu_gazebo_simulator bringup_sim.launch.py
```

### 10.4 手动控制机器人（底盘）
#### 用 `test_chassis_cmd`（键盘 → ChassisCmd）
（话题重映射到指定机器人 namespace）
```bash
source /opt/ros/humble/setup.bash
source /home/abc/rm_code/2026_1_21/ros2_ws/install/setup.bash
ros2 run rmoss_gz_base test_chassis_cmd --ros-args -r chassis_cmd:=/red_standard_robot1/robot_base/chassis_cmd
```

#### 直接发 `cmd_vel`（Twist）
前提：`rmua19_robot_base` 订阅到了该 namespace 下的 `cmd_vel`。
```bash
ros2 topic pub -r 10 /red_standard_robot1/cmd_vel geometry_msgs/msg/Twist "{linear: {x: 0.5, y: 0.0, z: 0.0}, angular: {x: 0.0, y: 0.0, z: 0.0}}"
```

### 10.5 常见排障清单（GUI 相关）
- 确认 `DISPLAY` 存在：`echo $DISPLAY`（你当前是 `:0`）
- 查 `log/launch_wrapper_gazebo_sim.log` 是否持续增长
- 如果你用远程/无桌面环境：需要 X11/Wayland 转发，否则 GUI 不可能弹